# RSNA Pneumonia Detection - Single Image Inference

This notebook loads a trained Faster R-CNN model and predicts pneumonia bounding boxes on a single DICOM chest X-ray image.

**Usage:** Just change `CHECKPOINT_PATH` and `IMAGE_PATH` in Cell 2, then run all cells.

In [1]:
import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import pydicom
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## Cell 1: Imports

## Cell 2: Configuration - CHANGE THESE PATHS

In [2]:
# ================= CHANGE THESE =================
CHECKPOINT_PATH = "output/checkpoints/checkpoint_epoch_2.pth"  # Your best model
IMAGE_PATH = "stage_2_train_images/0004cfab-14fd-4e18-8b82-9332f25b74d2.dcm"  # Change to any .dcm file
# =================================================

# Model settings
NUM_CLASSES = 2  # background + pneumonia
IMAGE_SIZE = 512
ORIGINAL_SIZE = 1024
SCORE_THRESHOLD = 0.5  # Minimum confidence to show a box

# ImageNet normalization (same as training)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Image: {IMAGE_PATH}")
print(f"Score threshold: {SCORE_THRESHOLD}")

Checkpoint: output/checkpoints/checkpoint_epoch_2.pth
Image: stage_2_train_images/0004cfab-14fd-4e18-8b82-9332f25b74d2.dcm
Score threshold: 0.5


## Cell 3: Model Loading Function

In [3]:
def load_model(checkpoint_path, num_classes=2, device='cuda'):
    """Load trained Faster R-CNN model from checkpoint."""
    
    # Create model architecture (same as training)
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(
        weights=weights,
        trainable_backbone_layers=3
    )
    
    # Replace box predictor for 2 classes (background + pneumonia)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Load checkpoint
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Handle both full checkpoint and state_dict-only formats
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            epoch = checkpoint.get('epoch', 'unknown')
            val_loss = checkpoint.get('val_loss', 'unknown')
            print(f"Loaded checkpoint from Epoch {epoch}, Val Loss: {val_loss}")
        else:
            model.load_state_dict(checkpoint)
            print("Loaded model state_dict")
    else:
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    model.to(device)
    model.eval()  # Set to evaluation mode
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model loaded: {total_params:,} total params, {trainable_params:,} trainable")
    
    return model

# Load the model
model = load_model(CHECKPOINT_PATH, NUM_CLASSES, device)

Loaded checkpoint from Epoch 2, Val Loss: 0.07256845912534769
Model loaded: 41,299,161 total params, 41,076,761 trainable


## Cell 4: DICOM Loading & Preprocessing

In [4]:
def load_dicom_image(dcm_path, image_size=512):
    """Load and preprocess a DICOM chest X-ray image."""
    
    # Load DICOM
    dcm = pydicom.dcmread(dcm_path)
    image = dcm.pixel_array
    
    # Store original for visualization
    original_image = image.copy()
    
    # Normalize to 0-255 and convert to uint8
    if image.dtype != np.uint8:
        image = image.astype(np.float32)
        image = (image - image.min()) / (image.max() - image.min() + 1e-8) * 255.0
        image = image.astype(np.uint8)
    
    # Convert grayscale to RGB
    if len(image.shape) == 2:
        image_rgb = np.stack([image] * 3, axis=-1)
    else:
        image_rgb = image
    
    # Convert to PIL Image for torchvision transforms
    image_pil = Image.fromarray(image_rgb)
    
    # Resize to model input size
    image_pil = image_pil.resize((image_size, image_size))
    
    # Convert to tensor and normalize with ImageNet stats
    image_tensor = torch.from_numpy(np.array(image_pil)).permute(2, 0, 1).float() / 255.0
    
    # Normalize
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    image_tensor = (image_tensor - mean) / std
    
    # Add batch dimension
    image_tensor = image_tensor.unsqueeze(0)
    
    return image_tensor, original_image

# Load the image
input_tensor, original_image = load_dicom_image(IMAGE_PATH, IMAGE_SIZE)
input_tensor = input_tensor.to(device)

print(f"Input tensor shape: {input_tensor.shape}")
print(f"Original image shape: {original_image.shape}")
print(f"Value range: [{original_image.min()}, {original_image.max()}]")

FileNotFoundError: [Errno 2] No such file or directory: 'stage_2_train_images/0004cfab-14fd-4e18-8b82-9332f25b74d2.dcm'

## Cell 5: Run Inference

In [ ]:
# Run inference (no gradient computation needed)
with torch.no_grad():
    predictions = model(input_tensor)

# Extract predictions for the first (and only) image in batch
pred = predictions[0]

boxes = pred['boxes'].cpu().numpy()
scores = pred['scores'].cpu().numpy()
labels = pred['labels'].cpu().numpy()

# Filter by confidence threshold
keep = scores >= SCORE_THRESHOLD
boxes = boxes[keep]
scores = scores[keep]
labels = labels[keep]

print(f"Total predictions before filtering: {len(pred['boxes'])}")
print(f"Predictions after threshold (>= {SCORE_THRESHOLD}): {len(boxes)}")
print()

if len(boxes) > 0:
    print("Detected pneumonia regions:")
    print("-" * 50)
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        print(f"  Box {i+1}: x={x1:.1f}, y={y1:.1f}, w={width:.1f}, h={height:.1f}")
        print(f"         Confidence: {score:.4f} ({score*100:.1f}%)")
        print()
else:
    print("No pneumonia detected above confidence threshold.")
    print("Try lowering SCORE_THRESHOLD in Cell 2 to see all predictions.")

## Cell 6: Visualization

In [ ]:
def visualize_predictions(original_image, boxes, scores, labels, score_threshold=0.5, figsize=(14, 14)):
    """Visualize original image with predicted bounding boxes."""
    
    # Prepare original image for display
    display_image = original_image.astype(np.float32)
    display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min())
    
    # Convert grayscale to RGB for display
    if len(display_image.shape) == 2:
        display_image = np.stack([display_image] * 3, axis=-1)
    
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(display_image, cmap='gray' if len(original_image.shape) == 2 else None)
    
    # Scale factor from model input size to original image size
    scale = original_image.shape[0] / IMAGE_SIZE
    
    # Draw boxes
    colors = ['#FF0000', '#FF6600', '#FFCC00', '#00FF00', '#00CCFF']
    
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        # Scale box coordinates back to original image size
        x1, y1, x2, y2 = box * scale
        width = x2 - x1
        height = y2 - y1
        
        color = colors[i % len(colors)]
        
        # Draw rectangle
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=3, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label with confidence
        label_text = f"Pneumonia: {score*100:.1f}%"
        ax.text(
            x1, y1 - 5,
            label_text,
            fontsize=12, color='white', weight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.8, edgecolor='white')
        )
    
    ax.set_title(
        f"RSNA Pneumonia Detection\n"
        f"{len(boxes)} region(s) detected (threshold: {score_threshold})",
        fontsize=14, weight='bold'
    )
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Visualize
if len(boxes) > 0:
    fig = visualize_predictions(original_image, boxes, scores, labels, SCORE_THRESHOLD)
else:
    # Show image even if no detections
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    display_image = original_image.astype(np.float32)
    display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min())
    ax.imshow(display_image, cmap='gray')
    ax.set_title("No pneumonia detected above threshold", fontsize=14)
    ax.axis('off')
    plt.show()

## Cell 7: Show All Predictions (Before Threshold)

In [ ]:
# Show ALL predictions (even low confidence) for analysis
all_boxes = pred['boxes'].cpu().numpy()
all_scores = pred['scores'].cpu().numpy()
all_labels = pred['labels'].cpu().numpy()

print(f"All predictions (no threshold): {len(all_boxes)}")
print("=" * 60)
for i, (box, score, label) in enumerate(zip(all_boxes, all_scores, all_labels)):
    x1, y1, x2, y2 = box
    above = "YES" if score >= SCORE_THRESHOLD else "no"
    print(f"  #{i+1}: score={score:.4f} [{above}] | box=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")

# Visualize with lower threshold to see all candidates
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Original image
display_image = original_image.astype(np.float32)
display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min())
if len(display_image.shape) == 2:
    display_image = np.stack([display_image] * 3, axis=-1)

scale = original_image.shape[0] / IMAGE_SIZE

# Left: All predictions
ax1.imshow(display_image)
for i, (box, score, label) in enumerate(zip(all_boxes, all_scores, all_labels)):
    x1, y1, x2, y2 = box * scale
    w, h = x2 - x1, y2 - y1
    alpha = min(score + 0.2, 1.0)
    color = 'red' if score >= SCORE_THRESHOLD else 'yellow'
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor='none', alpha=alpha)
    ax1.add_patch(rect)
    ax1.text(x1, y1 - 3, f"{score:.2f}", fontsize=9, color=color, weight='bold')
ax1.set_title(f"All Predictions (n={len(all_boxes)})", fontsize=12)
ax1.axis('off')

# Right: Filtered predictions
ax2.imshow(display_image)
for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
    x1, y1, x2, y2 = box * scale
    w, h = x2 - x1, y2 - y1
    rect = patches.Rectangle((x1, y1), w, h, linewidth=3, edgecolor='red', facecolor='none')
    ax2.add_patch(rect)
    ax2.text(x1, y1 - 5, f"Pneumonia: {score*100:.0f}%", fontsize=11, color='white',
             bbox=dict(boxstyle='round', facecolor='red', alpha=0.8))
ax2.set_title(f"Filtered (threshold={SCORE_THRESHOLD}, n={len(boxes)})", fontsize=12)
ax2.axis('off')

plt.suptitle("RSNA Pneumonia Detection - Prediction Comparison", fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

## Cell 8: Save Prediction Results

In [ ]:
# Save prediction results to a JSON-like structure
import json

results = {
    "image_path": IMAGE_PATH,
    "checkpoint": CHECKPOINT_PATH,
    "score_threshold": SCORE_THRESHOLD,
    "num_predictions": int(len(boxes)),
    "predictions": []
}

scale = original_image.shape[0] / IMAGE_SIZE

for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
    x1, y1, x2, y2 = box * scale  # Scale to original image coordinates
    results["predictions"].append({
        "box_id": i + 1,
        "x": float(x1),
        "y": float(y1),
        "width": float(x2 - x1),
        "height": float(y2 - y1),
        "confidence": float(score),
        "class": "pneumonia" if label == 1 else "background"
    })

# Save to file
output_file = "single_prediction_results.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_file}")
print("\nPrediction Summary:")
print(json.dumps(results, indent=2))

---

## How to use this notebook:

1. **Change paths in Cell 2:**
   - `CHECKPOINT_PATH`: Path to your trained model checkpoint
   - `IMAGE_PATH`: Path to any `.dcm` file you want to predict on

2. **Adjust threshold (optional):**
   - Lower `SCORE_THRESHOLD` (e.g., 0.3) to see more detections
   - Higher `SCORE_THRESHOLD` (e.g., 0.7) for only high-confidence predictions

3. **Run all cells** (Cell > Run All)

4. **Results:**
   - Predicted bounding boxes displayed on the image
   - Confidence scores for each detection
   - JSON file saved with coordinates

## Available Checkpoints:
- `output/checkpoints/checkpoint_epoch_2.pth` - **Best model** (val_loss: 0.0726)
- `output/checkpoints/checkpoint_epoch_5.pth` - Second best (val_loss: 0.0738)
- `output/best_model.pth` - Last saved best (Epoch 5)

## Model Performance (Epoch 2):
- Mean IoU: 0.6266
- AP@0.5: 0.4526
- mAP: 0.4441
- Precision: 0.3927
- Recall: 0.6569
- F1 Score: 0.4916